# Projecte Final: Anàlisi del finançament de les administracions públiques de Catalunya als mitjans de comunicació (2015-2024)

## Notebook 1 ETL-Extracció de dades — Menjòmetre (Premsa, 2015-2025)

**Com funciona:** La pàgina és Next.js. Les dades estan dins del bundleJavaScript amb cometes escapades com `\"`. Un sol `replace` les converteix
en JSON parsejable, i `json.loads()` extreu tots els camps de forma robusta.

Captura **totes** les entitats (individuals i de grup) amb el nom net. Cada entitat del JSON del Menjòmetre porta un camp `by_admin` que desglossa el seu import TOTAL per administració finançadora (Generalitat, diputacions, ajuntaments...), el captura i l'afegeix com a columnes noves.


In [1]:
import requests
import pandas as pd
from io import StringIO
import re
import json
import time

ANYS     = range(2015, 2025)
URL_BASE = "https://www.menjometre.cat"
HEADERS  = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}


### 1. Funcions

In [2]:
def descarregar_pagina(any_estudi):
    url = f'{URL_BASE}/premsa?year={any_estudi}'
    r = requests.get(url, headers=HEADERS, timeout=30)
    if r.status_code != 200:
        print(f'  Error {r.status_code}')
        return None
    return r.text ## retornem contingut de la pàgina



PATRO_ENTITAT = re.compile(
    r'\{"entity_id":"[^"]+".*?"by_admin":\{[^}]*\}',
    re.DOTALL
)


def extreure_entitats(text, any_estudi):
    """
    Extreu totes les entitats del text. En lloc de fer servir una llista
    fixa de claus d'administració (que pot quedar-se curta si el JSON
    font n'afegeix de noves), es guarda el diccionari `by_admin` SENCER
    tal qual a cada fila (camp '_by_admin_raw'). Les columnes admin_*
    es construeixen més endavant, un cop conegudes TOTES les claus que
    apareixen realment a tot el dataset (veure expandir_columnes_admin).
    """
    text_net = text.replace(chr(92) + chr(34), chr(34)) ##traiem cometes: equivalent a text.replace('\\"','"')

    entitats = []
    vistos = set()

    for m in PATRO_ENTITAT.finditer(text_net):
        bloc = m.group(0) + '}'  # tanquem l'objecte JSON

        try:
            d = json.loads(bloc)
        except json.JSONDecodeError:
            continue

        nom   = d.get('name', '')
        cif   = (d.get('cif') or '').upper()
        grup  = d.get('group_name') or ''
        subv  = float(d.get('subvencions', 0))
        cont  = float(d.get('contractes',  0))
        pub   = float(d.get('pub_inst',    0))
        total = float(d.get('total',       0))
        by_admin = d.get('by_admin', {}) or {}

        if not nom:
            continue

        clau = (cif or nom, round(total, 2))
        if clau in vistos:
            continue
        vistos.add(clau)

        fila = {
            'Any':                any_estudi,
            'Mitja':              nom,
            'CIF':                cif,
            'Grup':               grup,
            'Subvencions':        round(subv,  2),
            'Contractes':         round(cont,  2),
            'Pub. institucional': round(pub,   2),
            'Total':              round(total, 2),
            # Diccionari sencer, sense filtrar per cap llista fixa de claus
            '_by_admin_raw': {k: round(float(v), 2) for k, v in by_admin.items()},
        }

        entitats.append(fila)

    return entitats


def expandir_columnes_admin(df):
    """
    Converteix la columna '_by_admin_raw' (diccionaris) en columnes
    admin_{clau} planes, detectant dinàmicament TOTES les claus que
    apareixen a qualsevol fila del dataset (no una llista fixa).
    Garanteix suma(admin_*) == Total sempre, ja que es fa servir
    directament el contingut real de by_admin sense descartar cap clau.
    """
    totes_les_claus = sorted({
        clau
        for diccionari in df['_by_admin_raw']
        for clau in diccionari.keys()
    })
    print(f"Claus d'administració detectades dinàmicament ({len(totes_les_claus)}): {totes_les_claus}")

    for clau_admin in totes_les_claus:
        df[f'admin_{clau_admin}'] = df['_by_admin_raw'].apply(
            lambda d: d.get(clau_admin, 0.0)
        )

    df = df.drop(columns=['_by_admin_raw'])
    return df


print('Funcions definides OK')


Funcions definides OK


### 2. Test 2024 — verificació que s'ha fet l'extracció correctament

In [3]:
print('Test any 2024...')
html = descarregar_pagina(2024)

if html:
    ents = extreure_entitats(html, 2024)
    df_test = pd.DataFrame(ents)
    print(f'Entitats trobades: {len(df_test)}')
    print()

    # Expandir les columnes admin_* dinàmicament (detecta totes les claus)
    df_test = expandir_columnes_admin(df_test)
    print()

    print('La Vanguardia — desglossament per administració:')
    fila_lv = df_test[df_test['Mitja'].str.contains('VANGUARDIA', case=False, na=False)]
    display(fila_lv)
    print()

    # Verificar que la suma de columnes admin_* coincideix amb Total
    cols_admin = [c for c in df_test.columns if c.startswith('admin_')]
    df_test['suma_admin'] = df_test[cols_admin].sum(axis=1)
    df_test['diferencia'] = (df_test['suma_admin'] - df_test['Total']).abs()
    print(f'Verificació: diferència màxima entre suma admin_* i Total: '
          f'{df_test["diferencia"].max():.4f} €')
    print('(Hauria de ser 0 o molt propera a 0 -- amb detecció dinàmica ha de ser sempre 0)')
else:
    print('Error en descarregar la pagina.')

Test any 2024...
Entitats trobades: 176

Claus d'administració detectades dinàmicament (8): ['ajuntament_bcn', 'altres', 'altres_ajuntaments', 'diputacio_bcn', 'diputacio_girona', 'diputacio_lleida', 'diputacio_tarragona', 'generalitat']

La Vanguardia — desglossament per administració:


,Any,Mitja,CIF,Grup,Subvencions,Contractes,Pub. institucional,Total,admin_ajuntament_bcn,admin_altres,admin_altres_ajuntaments,admin_diputacio_bcn,admin_diputacio_girona,admin_diputacio_lleida,admin_diputacio_tarragona,admin_generalitat
1,2024,"LA VANGUARDIA EDICIONES,S.L.",B61475257,Grup Godó,982861.44,143292.37,2796033.61,3922187.42,25673.08,43770.4,30851.99,0.0,0.0,3765.77,0.0,3818126.18



Verificació: diferència màxima entre suma admin_* i Total: 0.0000 €
(Hauria de ser 0 o molt propera a 0 -- amb detecció dinàmica ha de ser sempre 0)


### 3. Fase 1 — Totals anuals (`df_totals`)

Fem extracció de dades per tenir una primera taula amb els totals anuals (2015-2025) de 'Subvencions', 'Contractes', 'Pub. institucional',

In [4]:
print('FASE 1 - Totals anuals')


html = descarregar_pagina(2024)
taules = pd.read_html(StringIO(html))

df_totals = taules[0].copy()
df_totals.columns = ['Any', 'Subvencions', 'Contractes', 'Pub. institucional', 'Total']
for col in ['Subvencions', 'Contractes', 'Pub. institucional', 'Total']:
    df_totals[col] = pd.to_numeric(df_totals[col], errors='coerce').fillna(0.0)

print(f'OK: {len(df_totals)} files')
display(df_totals)


FASE 1 - Totals anuals
OK: 11 files


,Any,Subvencions,Contractes,Pub. institucional,Total
0,2015,0,0,21516722,21516722
1,2016,0,3925,18632972,18636897
2,2017,0,206612,18723304,18929916
3,2018,11022,712266,17220167,17943455
4,2019,404182,2197214,19242143,21843538
5,2020,1918864,1472600,23583195,26974659
6,2021,7588141,2278762,23093431,32960335
7,2022,11482035,1873168,22631084,35986287
8,2023,1980765,8623189,27429242,38033196
9,2024,12874294,6710445,27641107,47225846


### 4. Fase 2 — Detall de mitjans (`df_detall`)

Fem extracció de dades per tenir una segona taula amb els totals anuals (2015-2024) de 'Subvencions', 'Contractes', 'Pub. institucional',

In [5]:
print('FASE 2 - Detall de mitjans (amb administració)')


llista = []

for any_estudi in ANYS:
    print(f'  Processant {any_estudi}...', end=' ')

    html = descarregar_pagina(any_estudi)
    if html is None:
        print('Error.')
        continue

    ents = extreure_entitats(html, any_estudi)
    llista.extend(ents)
    print(f'{len(ents)} entitats')
    time.sleep(1)

print()
print('Extraccio completada.')


FASE 2 - Detall de mitjans (amb administració)
  Processant 2015... 44 entitats
  Processant 2016... 43 entitats
  Processant 2017... 45 entitats
  Processant 2018... 53 entitats
  Processant 2019... 71 entitats
  Processant 2020... 90 entitats
  Processant 2021... 152 entitats
  Processant 2022... 161 entitats
  Processant 2023... 139 entitats
  Processant 2024... 176 entitats

Extraccio completada.


### 5. Construir `df_detall`

In [6]:
df_detall = pd.DataFrame(llista)

# Expandir les columnes admin_* dinàmicament sobre TOT el dataset
# (2015-2025 junts), perquè es detectin totes les claus que apareguin
# en qualsevol any, no només les del test de 2024.
df_detall = expandir_columnes_admin(df_detall)

cols_admin = [c for c in df_detall.columns if c.startswith('admin_')]
cols_finals = ['Any', 'Mitja', 'CIF', 'Grup',
               'Subvencions', 'Contractes', 'Pub. institucional', 'Total'] + cols_admin
df_detall = df_detall[cols_finals]

print(f'df_detall: {df_detall.shape[0]} files x {df_detall.shape[1]} columnes')
print()

print('Primeres 10 files (any 2024):')
display(df_detall[df_detall['Any'] == 2024].head(10))



Claus d'administració detectades dinàmicament (9): ['ajuntament_bcn', 'altres', 'altres_ajuntaments', 'diputacio_bcn', 'diputacio_girona', 'diputacio_lleida', 'diputacio_tarragona', 'estat', 'generalitat']
df_detall: 974 files x 17 columnes

Primeres 10 files (any 2024):


,Any,Mitja,CIF,Grup,Subvencions,Contractes,Pub. institucional,Total,admin_ajuntament_bcn,admin_altres,admin_altres_ajuntaments,admin_diputacio_bcn,admin_diputacio_girona,admin_diputacio_lleida,admin_diputacio_tarragona,admin_estat,admin_generalitat
798,2024,"CORPORACIO CATALANA DE MITJANS AUDIOVISUALS, S.A.",A08849622,CCMA (Mitjans públics catalans),0.00,159380.19,3775859.43,3935239.62,0.00,125076.72,19703.47,0.00,0.0,0.00,0.0,0.0,3790459.43
799,2024,"LA VANGUARDIA EDICIONES,S.L.",B61475257,Grup Godó,982861.44,143292.37,2796033.61,3922187.42,25673.08,43770.40,30851.99,0.00,0.0,3765.77,0.0,0.0,3818126.18
800,2024,"EL PERIODICO DE CATALUNYA, SLU",B66485343,Prensa Ibérica,703780.07,272872.38,2442227.82,3418880.27,34012.50,114899.96,72206.27,0.00,0.0,2942.31,0.0,0.0,3194819.23
801,2024,"EDICIO DE PREMSA PERIODICA ARA, SL",B65258261,Diari ARA,879502.50,315106.49,2202737.02,3397346.01,24378.61,124817.74,40412.52,8551.04,0.0,20941.54,0.0,0.0,3178244.56
802,2024,"RADIOCAT XXI,S.L.",B61726626,Grup Godó,749846.52,0.00,2149657.72,2899504.24,0.00,0.00,0.00,0.00,0.0,0.00,0.0,0.0,2899504.24
803,2024,"HERMES COMUNICACIONS,S.A.",A17374547,Hermes Comunicacions (Punt Avui),845763.84,400614.70,1227257.34,2473635.88,11389.04,58872.83,250475.49,304.55,0.0,11722.42,5272.3,0.0,2135599.25
804,2024,"SOCIEDAD ESPANOLA DE RADIODIFUSION, SL",B28016970,,200000.00,472962.14,1126347.79,1799309.93,0.00,263162.84,199882.30,0.00,9917.0,0.00,0.0,0.0,1326347.79
805,2024,"GRUP FLAIX, SL",B61801635,,340000.00,0.00,1112330.56,1452330.56,0.00,0.00,0.00,0.00,0.0,0.00,0.0,0.0,1452330.56
806,2024,"DIARI SEGRE, S.L.U.",B25275926,,317594.58,560210.62,408969.80,1286775.00,0.00,11115.90,172282.70,0.00,0.0,375373.54,0.0,0.0,728002.86
807,2024,GRUP LES NOTICIES DE CATALUNYA,B66541947,,504494.75,22000.00,663982.92,1190477.67,0.00,0.00,0.00,0.00,0.0,22000.00,0.0,0.0,1168477.67


### 6.  Verificació creuada 

Verificació creuada per comprovar que els totals del "df_totals" coincideixen amb les "df_detall" (l'any 2025 no s'ha extret en "df_detall" i sí en "df_totals")

In [7]:
suma = (df_detall.groupby("Any")["Total"]
             .sum().rename("Total (suma detall)").reset_index())
verif = df_totals[["Any", "Total"]].rename(
    columns={"Total": "Total (oficial)"}
).merge(suma, on="Any", how="left")
verif["Diferencia"] = (verif["Total (suma detall)"] - verif["Total (oficial)"]).round(0)
print("Totals oficials vs suma del detall:")
display(verif)


Totals oficials vs suma del detall:


,Any,Total (oficial),Total (suma detall),Diferencia
0,2015,21516722,21516721.66,-0.0
1,2016,18636897,18636897.32,0.0
2,2017,18929916,18929915.79,-0.0
3,2018,17943455,17943455.36,0.0
4,2019,21843538,21843537.90,-0.0
5,2020,26974659,26974658.85,-0.0
6,2021,32960335,32960334.55,-0.0
7,2022,35986287,35986286.57,-0.0
8,2023,38033196,38033195.50,-0.0
9,2024,47225846,47225845.98,-0.0


### 7. Exportació

Fem l'exportació a CSV dels dos Dataframes extrets.


In [8]:
df_totals.to_csv("menjometre_totals_anuals_admin.csv", index=False, encoding="utf-8-sig")
df_detall.to_csv("menjometre_detallat_admin.csv",       index=False, encoding="utf-8-sig")
print("Fitxers guardats:")
print("  - menjometre_totals_anuals_admin.csv")
print("  - menjometre_detallat_admin.csv")


Fitxers guardats:
  - menjometre_totals_anuals_admin.csv
  - menjometre_detallat_admin.csv
